In [1]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install git+https://github.com/ibm-granite-community/utils.git \
    transformers \
    'langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git' \
    docling
! echo "::endgroup::"

::group::Install Dependencies
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 63.5 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 112 packages in 6.09s
Prepared 32 packages in 2.57s
Uninstalled 3 packages in 739ms
Installed 32 packages in 275ms
 + colorlog==6.10.1
 + docling==2.81.0
 + docling-core==2.70.2
 + docling-ibm-models==3.12.0
 + docling-parse==5.6.1
 + faker==40.11.1
 + filetype==1.2.0
 - huggingface-hub==1.7.1
 + huggingface-hub==0.36.2
 + ibm-granite-community-utils==0.1.dev144 (from git+https://github.com/ibm-granite-community/utils.git@1a74b7b72ad70388df00acfb2e18b1f6592863c5)
 + jsonlines==4.0.0
 + jsonref==1.1.0
 + langchain-replicate==0.1.dev26 (from git+https://github.com/ibm-granite-community/langchain-replicate.git@200c6f94a8c3bb59afc5dda0dfd88490cd5ba952)
 + latex2mathml==3.79.0
 + marko==2.2.2
 + mpire==2.10.2
 + polyfactory==3.3.0
 + pyclipper==1.4.0
 + pylatexenc==2.10
 + pypdfium2==5.6.0
 + python-docx==1.2.0
 + python-pptx==1.0.2


In [6]:
from langchain_replicate import ChatReplicate
from ibm_granite_community.notebook_utils import get_env_var

model_path = "ibm-granite/granite-4.0-h-small"
model = ChatReplicate(
    model=model_path,
    replicate_api_token=get_env_var('token'),
    model_kwargs={
        "max_tokens": 2000, # Set the maximum number of tokens to generate as output.
        "min_tokens": 200, # Set the minimum number of tokens to generate as output.
        "presence_penalty": 0,
        "frequency_penalty": 0,
    },
)

token loaded from Google Colab secret.


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [8]:
import itertools
from typing import Iterator, Callable
from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.hierarchical_chunker import HierarchicalChunker
from docling_core.transforms.chunker.base import BaseChunk

def chunk_document(source: str, *, dropwhile: Callable[[BaseChunk], bool] = lambda c: False, takewhile: Callable[[BaseChunk], bool] = lambda c: True) -> Iterator[BaseChunk]:
    """Read the document and perform a hierarchical chunking"""
    converter = DocumentConverter()
    chunks = HierarchicalChunker().chunk(converter.convert(source=source).document)
    return itertools.takewhile(takewhile, itertools.dropwhile(dropwhile, chunks))

def merge_chunks(chunks: Iterator[BaseChunk], *, headings: Callable[[BaseChunk], list[str]] = lambda c: c.meta.headings) -> Iterator[dict[str, str]]:
    """Merge chunks having the same headings"""
    prior_headings: list[str] | None = None
    document: dict[str, str] = {}
    doc_id = 0
    for chunk in chunks:
        text = chunk.text.replace('\r\n', '\n')
        current_headings = headings(chunk)
        if prior_headings != current_headings:
            if document:
                yield document
            prior_headings = current_headings
            document = {
                'doc_id': str(doc_id:=doc_id+1),
                'title': " - ".join(current_headings),
                'text': text
            }
        else:
            document['text'] += f"\n\n{text}"
    if document:
        yield document

def chunk_dropwhile(chunk: BaseChunk) -> bool:
    """Ignore front matter prior to the book start"""
    return chunk.meta.headings != ["WALDEN", "Economy"]

def chunk_takewhile(chunk: BaseChunk) -> bool:
    """Ignore remaining chunks once we see this heading"""
    return "ON THE DUTY OF CIVIL DISOBEDIENCE" not in chunk.meta.headings

def chunk_headings(chunk: BaseChunk) -> list[str]:
    """Use the h1 and h2 (chapter) headings"""
    return chunk.meta.headings[:2]

documents: list[dict[str, str]] = list(merge_chunks(
    chunk_document(
        "https://www.gutenberg.org/cache/epub/205/pg205-images.html",
        dropwhile=chunk_dropwhile,
        takewhile=chunk_takewhile,
    ),
    headings=chunk_headings,
))

print(f"{len(documents)} documents created")
print(f"Max document size: {max(len(tokenizer.tokenize(document['text'])) for document in documents)} tokens")

18 documents created
Max document size: 33510 tokens


In [9]:
from ibm_granite_community.langchain.prompts import TokenizerChatPromptTemplate

prompt_template = TokenizerChatPromptTemplate.from_template("{user_prompt}", tokenizer=tokenizer)

def generate(user_prompt: str, documents: list[dict[str, str]]) -> str:
    """Use the chat template to format the prompt"""
    prompt = prompt_template.format_prompt(user_prompt=user_prompt, documents=documents)

    print(f"Input size: {len(tokenizer.tokenize(prompt.to_string()))} tokens")
    output = model.invoke(prompt, documents=documents)
    print(f"Output size: {len(tokenizer.tokenize(output.text))} tokens")

    return output.text

In [14]:
if get_env_var('GRANITE_TESTING', 'false').lower() == 'true':
    documents = documents[:5] # shorten testing work

user_prompt = """\
Using only the the book chapter document, compose a summary of the book chapter.
Your response should only include the summary. Do not provide any further explanation."""

summaries: list[dict[str, str]] = []

import time # Import the time module

for i, document in enumerate(documents, start=1):
    print(f"============================= {document['title']} ({i}/{len(documents)}) ===========================")
    output = generate(user_prompt, [document])
    summaries.append({
        'doc_id': document['doc_id'],
        'title': document['title'],
        'text': output,
    })
    time.sleep(10) # Increase the delay to 10 seconds to respect rate limits

print("Summary count: " + str(len(summaries)))

============================= WALDEN - Economy (1/18) ===========================
Input size: 36076 tokens
Output size: 235 tokens
============================= WALDEN - Where I Lived, and What I Lived For (2/18) ===========================
Input size: 8626 tokens
Output size: 361 tokens
============================= WALDEN - Reading (3/18) ===========================
Input size: 5402 tokens
Output size: 206 tokens
============================= WALDEN - Sounds (4/18) ===========================
Input size: 8374 tokens
Output size: 220 tokens
============================= WALDEN - Solitude (5/18) ===========================
Input size: 4905 tokens
Output size: 217 tokens
============================= WALDEN - Visitors (6/18) ===========================
Input size: 6770 tokens
Output size: 222 tokens
============================= WALDEN - The Bean-Field (7/18) ===========================
Input size: 5850 tokens
Output size: 207 tokens
============================= WALDEN - The Village (8

In [15]:
from ibm_granite_community.notebook_utils import wrap_text

user_prompt = """\
Using only the book chapter summary documents, compose a single, unified summary of the book.
Your response should only include the unified summary. Do not provide any further explanation."""

output = generate(user_prompt, summaries)
print(wrap_text(output))

Input size: 4718 tokens
Output size: 736 tokens
"Walden" by Henry David Thoreau is a reflection on simple living in natural
surroundings. The book is divided into several chapters, each focusing on a
different aspect of Thoreau's experiences living in a cabin near Walden Pond. In
"Economy," Thoreau describes his two-year experiment of living simply and self-
sufficiently, detailing the costs of building his cabin and maintaining his
lifestyle. He argues that a simpler life can lead to greater happiness and
freedom, and encourages readers to question their own values and priorities.
"Where I Lived, and What I Lived For" reflects on Thoreau's experiences living
in the woods, emphasizing the beauty and simplicity of nature and the importance
of living deliberately and mindfully. "Reading" emphasizes the importance of
reading and studying classic literature, arguing that it provides a deeper
understanding of human nature and history. "Sounds" describes Thoreau's
experience of living near W